## Lab: Creating DataFrames from Multiple Sources and Querying with Spark SQL

In [30]:
# Initialize SparkSession
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("SparkSQLLab").getOrCreate()
print("SparkSession initialized:")
spark

SparkSession initialized:


### Phase 1 – Prepare Source Files

In [31]:
# CSV – Customer Data
!echo -e "cust_id,name,city\n1,Alice,New York\n2,Bob,Los Angeles\n3,Charlie,Chicago\n4,Diana,Houston" > customers.csv

In [32]:
!echo '{"txn_id":101,"cust_id":1,"amount":250.5,"category":"Electronics"}' > transactions.json
!echo '{"txn_id":102,"cust_id":2,"amount":100.0,"category":"Books"}' >> transactions.json
!echo '{"txn_id":103,"cust_id":3,"amount":500.0,"category":"Fashion"}' >> transactions.json
print("transactions.json created.")

transactions.json created.


In [33]:
# Parquet – Product Data
products_data = [
    (1, "Laptop", "Electronics", 800),
    (2, "Book", "Books", 20),
    (3, "Shirt", "Fashion", 35)
]
df_products = spark.createDataFrame(products_data, ["prod_id", "name", "category", "price"])
df_products.write.mode("overwrite").parquet("products.parquet")
print("products.parquet created.")

[Stage 0:>                                                          (0 + 2) / 2]

products.parquet created.


### Phase 2 – Load Data into Spark DataFrames

In [34]:
# Load CSV:
custDF = spark.read.option("header", True).csv("customers.csv")
print("Customer DataFrame:")
custDF.show()

Customer DataFrame:
+-------+-------+-----------+
|cust_id|   name|       city|
+-------+-------+-----------+
|      1|  Alice|   New York|
|      2|    Bob|Los Angeles|
|      3|Charlie|    Chicago|
|      4|  Diana|    Houston|
+-------+-------+-----------+



In [35]:
# Load JSON:
txnDF = spark.read.json("transactions.json")
print("Transaction DataFrame:")
txnDF.show()

Transaction DataFrame:
+------+-----------+-------+------+
|amount|   category|cust_id|txn_id|
+------+-----------+-------+------+
| 250.5|Electronics|      1|   101|
| 100.0|      Books|      2|   102|
| 500.0|    Fashion|      3|   103|
+------+-----------+-------+------+



In [36]:
# Load Parquet:
prodDF = spark.read.parquet("products.parquet")
print("Product DataFrame:")
prodDF.show()

Product DataFrame:
+-------+------+-----------+-----+
|prod_id|  name|   category|price|
+-------+------+-----------+-----+
|      1|Laptop|Electronics|  800|
|      2|  Book|      Books|   20|
|      3| Shirt|    Fashion|   35|
+-------+------+-----------+-----+



### Phase 3 – Register DataFrames as Temporary Views

In [37]:
# Register each DataFrame for SQL querying:
custDF.createOrReplaceTempView("customers")
txnDF.createOrReplaceTempView("transactions")
prodDF.createOrReplaceTempView("products")
print("DataFrames registered as temporary views: customers, transactions, products")

DataFrames registered as temporary views: customers, transactions, products


### Phase 4 – Explore Each Dataset with SQL

In [38]:
# View all tables:
print("Available tables:")
spark.catalog.listTables()

Available tables:


[Table(name='customers', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='products', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='transactions', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

In [39]:
# Run basic SQL queries:
print("SELECT * FROM customers:")
spark.sql("SELECT * FROM customers").show()

print("SELECT name, city FROM customers WHERE city='New York':")
spark.sql("SELECT name, city FROM customers WHERE city='New York'").show()

print("SELECT category, COUNT(*) AS txn_count FROM transactions GROUP BY category:")
spark.sql("SELECT category, COUNT(*) AS txn_count FROM transactions GROUP BY category").show()

SELECT * FROM customers:
+-------+-------+-----------+
|cust_id|   name|       city|
+-------+-------+-----------+
|      1|  Alice|   New York|
|      2|    Bob|Los Angeles|
|      3|Charlie|    Chicago|
|      4|  Diana|    Houston|
+-------+-------+-----------+

SELECT name, city FROM customers WHERE city='New York':
+-----+--------+
| name|    city|
+-----+--------+
|Alice|New York|
+-----+--------+

SELECT category, COUNT(*) AS txn_count FROM transactions GROUP BY category:
+-----------+---------+
|   category|txn_count|
+-----------+---------+
|    Fashion|        1|
|Electronics|        1|
|      Books|        1|
+-----------+---------+



### Phase 5 – Perform Joins Across Data Sources

In [40]:
# Join customers and transactions:
print("Join customers and transactions:")
spark.sql("""
SELECT c.name, c.city, t.amount, t.category
FROM customers c
JOIN transactions t
ON c.cust_id = t.cust_id
""").show()

Join customers and transactions:
+-------+-----------+------+-----------+
|   name|       city|amount|   category|
+-------+-----------+------+-----------+
|  Alice|   New York| 250.5|Electronics|
|    Bob|Los Angeles| 100.0|      Books|
|Charlie|    Chicago| 500.0|    Fashion|
+-------+-----------+------+-----------+



In [41]:
# Join with products for category-level insights:
print("Join with products for category-level insights:")
spark.sql("""
SELECT c.name AS customer, c.city, t.category, p.name AS product, p.price, t.amount
FROM customers c
JOIN transactions t ON c.cust_id = t.cust_id
JOIN products p ON t.category = p.category
ORDER BY t.amount DESC
""").show()

Join with products for category-level insights:
+--------+-----------+-----------+-------+-----+------+
|customer|       city|   category|product|price|amount|
+--------+-----------+-----------+-------+-----+------+
| Charlie|    Chicago|    Fashion|  Shirt|   35| 500.0|
|   Alice|   New York|Electronics| Laptop|  800| 250.5|
|     Bob|Los Angeles|      Books|   Book|   20| 100.0|
+--------+-----------+-----------+-------+-----+------+



### Phase 6 – Run Aggregations and Filters

In [42]:
# Find top customers by spending:
print("Top customers by spending:")
spark.sql("""
SELECT c.name, SUM(t.amount) AS total_spent
FROM customers c
JOIN transactions t ON c.cust_id = t.cust_id
GROUP BY c.name
ORDER BY total_spent DESC
""").show()

Top customers by spending:
+-------+-----------+
|   name|total_spent|
+-------+-----------+
|Charlie|      500.0|
|  Alice|      250.5|
|    Bob|      100.0|
+-------+-----------+



### Phase 7 – Query Optimization and Caching

In [43]:
# Cache the joined view for repeated queries:
spark.sql("""
SELECT c.name, t.amount, t.category
FROM customers c
JOIN transactions t
ON c.cust_id = t.cust_id
""").createOrReplaceTempView("cust_txn_view")

spark.catalog.cacheTable("cust_txn_view")
print("cust_txn_view created and cached.")
print("Count from cached view:")
spark.sql("SELECT COUNT(*) FROM cust_txn_view").show()

cust_txn_view created and cached.
Count from cached view:
+--------+
|count(1)|
+--------+
|       3|
+--------+



### Phase 8 – Use SQL Expressions and Functions

In [44]:
# Example of SQL expressions:
print("Applying SQL expressions:")
spark.sql("""
SELECT c.name,
       t.amount,
       ROUND(t.amount * 0.10, 2) AS tax,
       t.amount + ROUND(t.amount * 0.10, 2) AS total_with_tax
FROM customers c
JOIN transactions t ON c.cust_id = t.cust_id
""").show()

Applying SQL expressions:
+-------+------+-----+--------------+
|   name|amount|  tax|total_with_tax|
+-------+------+-----+--------------+
|  Alice| 250.5|25.05|        275.55|
|    Bob| 100.0| 10.0|         110.0|
|Charlie| 500.0| 50.0|         550.0|
+-------+------+-----+--------------+



### Phase 9 – Save Results as Parquet for Reporting

In [45]:
# Write query results back to disk:
result = spark.sql("""
SELECT c.name, c.city, SUM(t.amount) AS total_spent
FROM customers c
JOIN transactions t ON c.cust_id = t.cust_id
GROUP BY c.name, c.city
""")
result.write.mode("overwrite").parquet("sales_report")
print("sales_report.parquet created.")

sales_report.parquet created.


### Phase 10 – Verify Output and Clean Up

In [46]:
# Check saved data:
print("Contents of sales_report.parquet:")
spark.read.parquet("sales_report").show()

Contents of sales_report.parquet:
+-------+-----------+-----------+
|   name|       city|total_spent|
+-------+-----------+-----------+
|    Bob|Los Angeles|      100.0|
|  Alice|   New York|      250.5|
|Charlie|    Chicago|      500.0|
+-------+-----------+-----------+



In [47]:
# Clean up (optional):
!rm -f customers.csv
!rm -f transactions.json
!rm -rf products.parquet
!rm -rf sales_report
print("Clean up complete.")

# Stop SparkSession
spark.stop()
print("SparkSession stopped.")

Clean up complete.
SparkSession stopped.
